# Ευανάγνωστοι κανονικοποιημένοι πίνακες σύγχυσης

Το notebook δημιουργεί πίνακες σύγχυσης με κοινή μορφοποίηση για το παράρτημα. Χρησιμοποιεί τα αρχεία με τους απόλυτους αριθμούς (`confusion_matrix_counts.csv`), κάνει αυτόματα την κανονικοποίηση ανά πραγματική κλάση και αποθηκεύει κάθε πίνακα σε PNG υψηλής ανάλυσης και σε SVG.

In [ ]:
from pathlib import Path
import re
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter


# Αν το repository έχει γίνει clone στο Kaggle, χρησιμοποιείται αυτόματα.
REPO_DIR = Path("/kaggle/working/thesis")

# Για εκτέλεση από τον τοπικό φάκελο του repository.
if not (REPO_DIR / "results").exists():
    REPO_DIR = Path.cwd()

if not (REPO_DIR / "results").exists():
    raise FileNotFoundError(
        "Δεν βρέθηκε ο φάκελος results. "
        "Τρέξε πρώτα το notebook μέσα από το repository thesis."
    )


if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/confusion_matrices_appendix")
else:
    OUTPUT_DIR = REPO_DIR / "confusion_matrices_appendix"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Ελάχιστο ποσοστό για εμφάνιση αριθμού εκτός της διαγωνίου.
OFF_DIAGONAL_LABEL_THRESHOLD = 0.05

print("Repository:", REPO_DIR)
print("Output:", OUTPUT_DIR)

## Επιλογή μοντέλων

Στο `MODEL_PATHS` υπάρχουν όλες οι τελικές εκδοχές για τις οποίες έχει αποθηκευτεί πίνακας σύγχυσης. Στη λίστα `SELECTED_MODELS` κρατάμε μόνο όσες θέλουμε να δημιουργηθούν. Η προεπιλογή καλύπτει τις τέσσερις βασικές αρχιτεκτονικές και τις δύο εκδοχές του MobileNetV1 που συγκρίνονται αναλυτικά στην εργασία.

In [ ]:
MODEL_PATHS = {
    "Custom CNN": (
        "results/custom_cnn/"
        "custom_cnn_clean_split_lanczos_seed12345_results/"
        "confusion_matrix_counts.csv"
    ),
    "Custom CNN + oversampling": (
        "results/custom_cnn/"
        "custom_cnn_clean_split_lanczos_oversampling_to100_seed12345_results/"
        "confusion_matrix_counts.csv"
    ),
    "EfficientNetB2 frozen": (
        "results/efficientnetb2/efficientnetb2_frozen_backbone/"
        "confusion_matrix_counts.csv"
    ),
    "EfficientNetB2 BN-locked": (
        "results/efficientnetb2/efficientnetb2_bn_locked_results/"
        "confusion_matrix_counts.csv"
    ),
    "EfficientNetB2 BN-locked + oversampling": (
        "results/efficientnetb2/efficientnetb2_bn_locked_oversampling_results/"
        "confusion_matrix_counts.csv"
    ),
    "EfficientNetB2 full unfreeze": (
        "results/efficientnetb2/efficientnetb2_clean_split_full_unfreeze_results/"
        "confusion_matrix_counts.csv"
    ),
    "EfficientNetB2 from scratch": (
        "results/efficientnetb2/"
        "efficientnetb2_clean_split_from_scratch_seed12345_results/"
        "confusion_matrix_counts.csv"
    ),
    "MobileNetV1 frozen": (
        "results/mobilenet/mobilenet_frozen_backbone_results/"
        "confusion_matrix_counts.csv"
    ),
    "MobileNetV1 BN-locked": (
        "results/mobilenet/mobilenet_clean_split_bn_locked_results/"
        "confusion_matrix_counts.csv"
    ),
    "MobileNetV1 BN-locked + oversampling": (
        "results/mobilenet/mobilenet_bn_locked_oversampling_results/"
        "confusion_matrix_counts.csv"
    ),
    "MobileNetV1 full unfreeze": (
        "results/mobilenet/mobilenet_clean_split_full_unfreeze_results/"
        "confusion_matrix_counts.csv"
    ),
    "MobileNetV1 from scratch": (
        "results/mobilenet/"
        "mobilenet_clean_split_from_scratch_seed12345_results/"
        "confusion_matrix_counts.csv"
    ),
    "ConvNeXtBase": (
        "results/convnextbase/full_finetune_test_confusion_matrix.csv"
    ),
}


# Αυτές οι εικόνες θα δημιουργηθούν. Πρόσθεσε ή αφαίρεσε ονόματα αν χρειάζεται.
SELECTED_MODELS = [
    "Custom CNN + oversampling",
    "EfficientNetB2 BN-locked",
    "MobileNetV1 frozen",
    "MobileNetV1 BN-locked",
    "ConvNeXtBase",
]


for model_name in SELECTED_MODELS:
    if model_name not in MODEL_PATHS:
        raise KeyError(f"Άγνωστο μοντέλο: {model_name}")

    counts_path = REPO_DIR / MODEL_PATHS[model_name]
    if not counts_path.exists():
        raise FileNotFoundError(f"Δεν βρέθηκε: {counts_path}")

print("Θα δημιουργηθούν", len(SELECTED_MODELS), "πίνακες:")
for model_name in SELECTED_MODELS:
    print("-", model_name)

## Δημιουργία των πινάκων

Οι κλάσεις ταξινομούνται με βάση το πλήθος εικόνων στο test set. Στον κατακόρυφο άξονα εμφανίζεται και το πλήθος των εικόνων κάθε κλάσης. Στα κελιά εμφανίζονται οι τιμές της διαγωνίου και οι λανθασμένες ταξινομήσεις με ποσοστό τουλάχιστον 5%.

In [ ]:
def safe_filename(text):
    filename = re.sub(r"[^A-Za-z0-9]+", "_", text)
    return filename.strip("_").lower()


def format_x_label(class_name):
    # Οι αλλαγές γραμμής βοηθούν τις μεγάλες ονομασίες να χωρέσουν.
    return class_name.replace("_", "\n")


def format_y_label(class_name, number_of_images):
    readable_name = class_name.replace("_", " ")
    return f"{readable_name} (n={int(number_of_images)})"


def create_confusion_matrix(model_name, relative_counts_path):
    counts_path = REPO_DIR / relative_counts_path
    counts = pd.read_csv(counts_path, index_col=0)

    # Σε ορισμένα αρχεία οι γραμμές περιέχουν ήδη το support, π.χ. "dinobryon (510)".
    counts.index = counts.index.astype(str).str.replace(
        r"\s+\(\d+\)$",
        "",
        regex=True,
    )
    counts.columns = counts.columns.astype(str).str.replace(
        r"\s+\(\d+\)$",
        "",
        regex=True,
    )

    # Ενοποιούμε την παλιά και τη διορθωμένη ορθογραφία της κλάσης.
    counts = counts.rename(
        index={"kellikottia": "kellicottia"},
        columns={"kellikottia": "kellicottia"},
    )

    counts = counts.apply(pd.to_numeric)
    support = counts.sum(axis=1).sort_values(ascending=False)
    class_names = support.index.tolist()

    missing_columns = [
        class_name
        for class_name in class_names
        if class_name not in counts.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Λείπουν στήλες από το {model_name}: {missing_columns}"
        )

    counts = counts.loc[class_names, class_names]

    # Κανονικοποίηση κάθε γραμμής με βάση τις εικόνες της πραγματικής κλάσης.
    matrix = counts.div(counts.sum(axis=1), axis=0).fillna(0)

    x_labels = [format_x_label(name) for name in class_names]
    y_labels = [
        format_y_label(name, support[name])
        for name in class_names
    ]

    fig, ax = plt.subplots(figsize=(19, 14))

    image = ax.imshow(
        matrix,
        cmap="Blues",
        vmin=0,
        vmax=1,
        interpolation="nearest",
        aspect="auto",
    )

    positions = np.arange(len(class_names))
    ax.set_xticks(positions, x_labels, rotation=90, fontsize=9)
    ax.set_yticks(positions, y_labels, fontsize=9)
    ax.set_xlabel("Προβλεπόμενη κλάση", fontsize=12, labelpad=10)
    ax.set_ylabel("Πραγματική κλάση", fontsize=12, labelpad=10)
    ax.set_title(model_name, fontsize=15, pad=14)

    # Λεπτό λευκό πλέγμα για να ξεχωρίζουν τα κελιά.
    ax.set_xticks(np.arange(-0.5, len(class_names), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(class_names), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.35, alpha=0.75)
    ax.tick_params(which="minor", bottom=False, left=False)

    colorbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.032,
        pad=0.02,
    )
    colorbar.set_label(
        "Ποσοστό εικόνων ανά πραγματική κλάση",
        fontsize=11,
    )
    colorbar.ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    colorbar.ax.tick_params(labelsize=9)

    for row in range(len(class_names)):
        for column in range(len(class_names)):
            value = matrix.iloc[row, column]

            if row == column or value >= OFF_DIAGONAL_LABEL_THRESHOLD:
                ax.text(
                    column,
                    row,
                    f"{value:.0%}",
                    ha="center",
                    va="center",
                    fontsize=6.2,
                    color="white" if value >= 0.50 else "black",
                )

    fig.tight_layout()

    filename = safe_filename(model_name)
    png_path = OUTPUT_DIR / f"{filename}.png"
    svg_path = OUTPUT_DIR / f"{filename}.svg"

    fig.savefig(
        png_path,
        dpi=350,
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        svg_path,
        format="svg",
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)

    return png_path, svg_path


created_files = []

for model_name in SELECTED_MODELS:
    png_path, svg_path = create_confusion_matrix(
        model_name,
        MODEL_PATHS[model_name],
    )
    created_files.extend([png_path, svg_path])
    print("Saved:", png_path.name, "and", svg_path.name)

print("\nΟλοκληρώθηκαν", len(SELECTED_MODELS), "πίνακες σύγχυσης.")

## Δημιουργία ZIP

Το τελευταίο κελί συγκεντρώνει όλες τις εικόνες σε ένα αρχείο ZIP, ώστε να κατέβουν εύκολα από το Kaggle. Για εισαγωγή στο Word προτίμησε τα αρχεία SVG. Αν η έκδοση του Word δεν τα εμφανίζει σωστά, χρησιμοποίησε τα PNG.

In [ ]:
zip_base = OUTPUT_DIR.parent / "confusion_matrices_appendix"
zip_path = Path(
    shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=OUTPUT_DIR,
    )
)

print("ZIP:", zip_path)